# SQL Chatbot - Complete Jupyter Notebook

This notebook provides a complete, self-contained implementation of the SQL Chatbot that can run entirely online in Jupyter environments (Google Colab, JupyterHub, etc.).

## Features
- Natural language to SQL query generation
- Data profiling and quality analysis
- Interactive chat interface
- Works with SQLite (no external database required)
- Uses Ollama for LLM capabilities


In [ ]:
# Install required packages
!pip install -q pandas requests

# Note: sqlite3 is part of Python's standard library, no need to install
# Note: For online Jupyter environments, you may need to use a different LLM service
# This notebook is designed to work with Ollama, but can be adapted for other services


## Step 1: Configuration Setup


In [ ]:
import os
import sys
import sqlite3
import logging
import re
import requests
from typing import Optional, List, Dict, Any, Tuple
from datetime import datetime, timedelta
from contextlib import contextmanager
from pydantic import BaseModel

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuration
class Config:
    """Configuration for the chatbot."""
    # Database settings - using SQLite for online compatibility
    db_path = "profiling_sample.db"
    
    # Ollama settings (adjust if using a different service)
    ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    ollama_model = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
    
    # Allowed database views (security constraint)
    allowed_views = [
        "profiling_column_stats",
        "profiling_table_stats",
        "profiling_data_quality"
    ]
    
    log_level = "INFO"

config = Config()
print("✅ Configuration loaded")


## Step 2: Database Setup


In [ ]:
def create_sample_database():
    """Create SQLite database with sample profiling data."""
    # Remove existing database if it exists
    if os.path.exists(config.db_path):
        os.remove(config.db_path)
        print(f"Removed existing database: {config.db_path}")
    
    conn = sqlite3.connect(config.db_path)
    conn.row_factory = sqlite3.Row  # Enable dict-like access
    cursor = conn.cursor()
    
    print("Creating database schema...")
    
    # Create base profiling tables
    cursor.execute("""
        CREATE TABLE profiling_tables (
            table_id INTEGER PRIMARY KEY AUTOINCREMENT,
            table_name TEXT NOT NULL,
            row_count INTEGER,
            column_count INTEGER,
            last_profiled_date TIMESTAMP,
            table_size_mb REAL
        )
    """)
    
    cursor.execute("""
        CREATE TABLE profiling_columns (
            column_id INTEGER PRIMARY KEY AUTOINCREMENT,
            table_id INTEGER,
            column_name TEXT NOT NULL,
            data_type TEXT,
            null_count INTEGER,
            distinct_count INTEGER,
            avg_length REAL,
            min_value TEXT,
            max_value TEXT,
            duplicate_count INTEGER,
            FOREIGN KEY (table_id) REFERENCES profiling_tables(table_id)
        )
    """)
    
    # Insert sample table data
    print("Inserting sample table data...")
    tables_data = [
        ("customers", 15000, 12, datetime.now() - timedelta(days=1), 2.5),
        ("orders", 45000, 8, datetime.now() - timedelta(days=2), 3.8),
        ("products", 2500, 15, datetime.now() - timedelta(days=1), 1.2),
        ("transactions", 125000, 10, datetime.now() - timedelta(hours=12), 8.5),
        ("employees", 500, 20, datetime.now() - timedelta(days=3), 0.8),
    ]
    
    for table_name, row_count, col_count, last_profiled, size_mb in tables_data:
        cursor.execute("""
            INSERT INTO profiling_tables (table_name, row_count, column_count, last_profiled_date, table_size_mb)
            VALUES (?, ?, ?, ?, ?)
        """, (table_name, row_count, col_count, last_profiled, size_mb))
    
    # Insert sample column data
    print("Inserting sample column profiling data...")
    
    # Customers table columns
    customers_columns = [
        (1, "customer_id", "INTEGER", 0, 15000, None, "1", "15000", 0),
        (1, "first_name", "VARCHAR", 0, 14200, 12.5, "Aaron", "Zoe", 800),
        (1, "last_name", "VARCHAR", 0, 14800, 15.2, "Anderson", "Zimmerman", 200),
        (1, "email", "VARCHAR", 1500, 13500, 24.8, "a@example.com", "z@example.com", 0),
        (1, "phone", "VARCHAR", 3200, 12000, 13.0, "100-000-0000", "999-999-9999", 3000),
        (1, "address", "VARCHAR", 800, 14500, 45.2, "100 Main St", "9999 Oak Ave", 500),
        (1, "city", "VARCHAR", 0, 850, 12.5, "Albany", "Zurich", 0),
        (1, "state", "VARCHAR", 0, 52, 2.0, "AK", "WY", 0),
        (1, "zip_code", "VARCHAR", 0, 1200, 5.0, "00001", "99999", 0),
        (1, "registration_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (1, "status", "VARCHAR", 0, 3, 6.0, "active", "inactive", 0),
        (1, "notes", "TEXT", 8500, 8500, 125.5, None, None, 0),
    ]
    
    # Orders table columns
    orders_columns = [
        (2, "order_id", "INTEGER", 0, 45000, None, "1", "45000", 0),
        (2, "customer_id", "INTEGER", 0, 12000, None, "1", "15000", 0),
        (2, "order_date", "DATE", 0, 1095, None, "2022-01-01", "2024-12-31", 0),
        (2, "total_amount", "DECIMAL", 0, 12500, None, "5.99", "9999.99", 0),
        (2, "status", "VARCHAR", 0, 5, 10.0, "pending", "shipped", 0),
        (2, "shipping_address", "VARCHAR", 500, 44000, 48.5, None, None, 1000),
        (2, "payment_method", "VARCHAR", 0, 8, 12.0, "cash", "wire_transfer", 0),
        (2, "notes", "TEXT", 35000, 10000, 85.2, None, None, 0),
    ]
    
    # Products table columns
    products_columns = [
        (3, "product_id", "INTEGER", 0, 2500, None, "1", "2500", 0),
        (3, "product_name", "VARCHAR", 0, 2500, 28.5, "Widget A", "Zebra Stripes", 0),
        (3, "category", "VARCHAR", 0, 25, 15.0, "Electronics", "Toys", 0),
        (3, "price", "DECIMAL", 0, 1250, None, "0.99", "999.99", 0),
        (3, "stock_quantity", "INTEGER", 0, 500, None, "0", "10000", 0),
        (3, "description", "TEXT", 200, 2300, 145.8, None, None, 0),
        (3, "supplier_id", "INTEGER", 500, 200, None, "1", "50", 0),
        (3, "created_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (3, "is_active", "BOOLEAN", 0, 2, None, "0", "1", 0),
        (3, "tags", "VARCHAR", 800, 1700, 25.5, None, None, 0),
        (3, "image_url", "VARCHAR", 1200, 1300, 45.2, None, None, 0),
        (3, "weight_kg", "DECIMAL", 300, 2200, None, "0.01", "50.00", 0),
        (3, "dimensions", "VARCHAR", 400, 2100, 18.5, None, None, 0),
        (3, "warranty_months", "INTEGER", 600, 25, None, "0", "60", 0),
        (3, "rating", "DECIMAL", 1500, 100, None, "1.0", "5.0", 0),
    ]
    
    # Transactions table columns
    transactions_columns = [
        (4, "transaction_id", "INTEGER", 0, 125000, None, "1", "125000", 0),
        (4, "order_id", "INTEGER", 0, 45000, None, "1", "45000", 0),
        (4, "transaction_date", "TIMESTAMP", 0, 87500, None, "2022-01-01 00:00:00", "2024-12-31 23:59:59", 0),
        (4, "amount", "DECIMAL", 0, 8750, None, "0.01", "5000.00", 0),
        (4, "currency", "VARCHAR", 0, 5, 3.0, "EUR", "USD", 0),
        (4, "payment_status", "VARCHAR", 0, 4, 10.0, "failed", "success", 0),
        (4, "processor_response", "TEXT", 25000, 100000, 125.5, None, None, 0),
        (4, "refund_amount", "DECIMAL", 110000, 1500, None, "0.00", "5000.00", 0),
        (4, "fraud_score", "DECIMAL", 50000, 75000, None, "0.0", "100.0", 0),
        (4, "metadata", "JSON", 30000, 95000, 85.2, None, None, 0),
    ]
    
    # Employees table columns
    employees_columns = [
        (5, "employee_id", "INTEGER", 0, 500, None, "1", "500", 0),
        (5, "first_name", "VARCHAR", 0, 485, 8.5, "Alice", "Zachary", 0),
        (5, "last_name", "VARCHAR", 0, 495, 10.2, "Adams", "Zimmer", 0),
        (5, "email", "VARCHAR", 0, 500, 22.5, "a.adams@company.com", "z.zimmer@company.com", 0),
        (5, "department", "VARCHAR", 0, 12, 15.0, "Engineering", "Sales", 0),
        (5, "position", "VARCHAR", 0, 45, 20.5, "Intern", "VP Engineering", 0),
        (5, "salary", "DECIMAL", 0, 450, None, "40000.00", "250000.00", 0),
        (5, "hire_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (5, "manager_id", "INTEGER", 50, 450, None, "1", "500", 0),
        (5, "phone_extension", "VARCHAR", 100, 400, 4.0, "1000", "9999", 0),
        (5, "office_location", "VARCHAR", 0, 8, 12.0, "Building A", "Remote", 0),
        (5, "emergency_contact", "VARCHAR", 200, 300, 35.5, None, None, 0),
        (5, "emergency_phone", "VARCHAR", 200, 300, 13.0, None, None, 0),
        (5, "start_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (5, "end_date", "DATE", 450, 50, None, None, None, 0),
        (5, "status", "VARCHAR", 0, 3, 8.0, "active", "terminated", 0),
        (5, "notes", "TEXT", 400, 100, 125.5, None, None, 0),
        (5, "performance_rating", "DECIMAL", 150, 350, None, "1.0", "5.0", 0),
        (5, "training_completed", "BOOLEAN", 0, 2, None, "0", "1", 0),
        (5, "certifications", "VARCHAR", 300, 200, 45.2, None, None, 0),
    ]
    
    all_columns = customers_columns + orders_columns + products_columns + transactions_columns + employees_columns
    
    cursor.executemany("""
        INSERT INTO profiling_columns 
        (table_id, column_name, data_type, null_count, distinct_count, avg_length, min_value, max_value, duplicate_count)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, all_columns)
    
    # Create views for the chatbot
    print("Creating profiling views...")
    
    cursor.execute("""
        CREATE VIEW profiling_column_stats AS
        SELECT 
            t.table_name,
            c.column_name,
            CAST(c.null_count * 100.0 / NULLIF(t.row_count, 0) AS REAL) AS null_percentage,
            c.distinct_count,
            c.data_type,
            t.row_count,
            c.avg_length,
            c.min_value,
            c.max_value
        FROM profiling_tables t
        INNER JOIN profiling_columns c ON t.table_id = c.table_id
    """)
    
    cursor.execute("""
        CREATE VIEW profiling_table_stats AS
        SELECT 
            table_name,
            row_count,
            column_count,
            last_profiled_date AS last_updated,
            table_size_mb
        FROM profiling_tables
    """)
    
    cursor.execute("""
        CREATE VIEW profiling_data_quality AS
        SELECT 
            t.table_name,
            c.column_name,
            CAST(c.null_count * 100.0 / NULLIF(t.row_count, 0) AS REAL) AS null_percentage,
            CAST(c.duplicate_count * 100.0 / NULLIF(t.row_count, 0) AS REAL) AS duplicate_percentage,
            CASE 
                WHEN c.null_count * 100.0 / NULLIF(t.row_count, 0) > 50 THEN 'Poor'
                WHEN c.null_count * 100.0 / NULLIF(t.row_count, 0) > 10 THEN 'Fair'
                ELSE 'Good'
            END AS quality_score
        FROM profiling_tables t
        INNER JOIN profiling_columns c ON t.table_id = c.table_id
    """)
    
    conn.commit()
    conn.close()
    
    print(f"\n✅ Database created successfully: {config.db_path}")
    print(f"✅ Inserted {len(tables_data)} tables")
    print(f"✅ Inserted {len(all_columns)} columns")
    print(f"✅ Created 3 profiling views")
    print("\nDatabase is ready for use!")

# Create the database
create_sample_database()


## Step 3: Core Components


In [ ]:
# Database Connection Class
class DatabaseConnection:
    """Manages database connections and query execution."""
    
    def __init__(self, db_path: str):
        self.db_path = db_path
    
    @contextmanager
    def get_connection(self):
        """Context manager for database connections."""
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row  # Enable dict-like access
        try:
            yield conn
        finally:
            conn.close()
    
    def execute_query(self, query: str) -> List[Dict[str, Any]]:
        """Execute a SELECT query and return results as a list of dictionaries."""
        with self.get_connection() as conn:
            cursor = conn.cursor()
            try:
                cursor.execute(query)
                columns = [description[0] for description in cursor.description]
                rows = cursor.fetchall()
                results = [dict(row) for row in rows]
                return results
            except Exception as e:
                logger.error(f"Query execution error: {e}")
                raise
            finally:
                cursor.close()

# Initialize database connection
db = DatabaseConnection(config.db_path)
print("✅ Database connection initialized")


In [ ]:
# SQL Validator Class
class SQLValidator:
    """Validates SQL queries to ensure they are safe and read-only."""
    
    DANGEROUS_KEYWORDS = [
        'INSERT', 'UPDATE', 'DELETE', 'DROP', 'CREATE', 'ALTER',
        'TRUNCATE', 'EXEC', 'EXECUTE', 'GRANT', 'REVOKE', 'MERGE'
    ]
    
    def __init__(self, allowed_views: List[str]):
        self.allowed_views = allowed_views
    
    def validate(self, query: str) -> Tuple[bool, Optional[str]]:
        """Validate a SQL query for safety."""
        if not query or not query.strip():
            return False, "Query is empty"
        
        query_upper = query.upper().strip()
        
        if not query_upper.startswith('SELECT'):
            return False, "Only SELECT queries are allowed"
        
        for keyword in self.DANGEROUS_KEYWORDS:
            pattern = r'\b' + re.escape(keyword) + r'\b'
            if re.search(pattern, query_upper):
                return False, f"Dangerous keyword '{keyword}' is not allowed"
        
        view_pattern = r'\bFROM\s+(\w+)\b'
        matches = re.findall(view_pattern, query_upper)
        
        if matches:
            referenced_views = [match.upper() for match in matches]
            allowed_upper = [view.upper() for view in self.allowed_views]
            
            for view in referenced_views:
                if view not in allowed_upper:
                    return False, f"View '{view}' is not in the allowed list"
        
        return True, None
    
    def sanitize(self, query: str) -> str:
        """Sanitize a SQL query by removing trailing semicolons and extra whitespace."""
        query = query.rstrip().rstrip(';')
        query = ' '.join(query.split())
        return query

sql_validator = SQLValidator(config.allowed_views)
print("✅ SQL validator initialized")


In [ ]:
# Ollama Client Class
class OllamaClient:
    """Client for interacting with Ollama API."""
    
    def __init__(self, base_url: str, model: str):
        self.base_url = base_url
        self.model = model
    
    def generate_sql(self, user_question: str) -> Optional[str]:
        """Generate SQL query from natural language question."""
        prompt = self._build_sql_prompt(user_question)
        
        try:
            # Check if model is available
            models_response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            if models_response.status_code == 200:
                models = models_response.json().get("models", [])
                model_names = [m.get("name", "") for m in models]
                if self.model not in model_names:
                    logger.warning(f"Model '{self.model}' not found. Available: {model_names}")
                    return None
            
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.1,
                        "top_p": 0.9,
                    }
                },
                timeout=120
            )
            response.raise_for_status()
            
            result = response.json()
            sql_query = result.get("response", "").strip()
            sql_query = self._extract_sql_from_response(sql_query)
            
            return sql_query
            
        except requests.exceptions.ConnectionError:
            logger.error(f"Cannot connect to Ollama at {self.base_url}. Is Ollama running?")
            return None
        except Exception as e:
            logger.error(f"Ollama API error: {e}")
            return None
    
    def summarize_results(self, question: str, results: list, explanation: str) -> str:
        """Generate a natural language summary of query results."""
        prompt = self._build_summary_prompt(question, results, explanation)
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.3,
                        "top_p": 0.9,
                    }
                },
                timeout=90
            )
            response.raise_for_status()
            
            result = response.json()
            return result.get("response", "").strip()
            
        except Exception as e:
            logger.error(f"Ollama API error during summarization: {e}")
            return explanation
    
    def _build_sql_prompt(self, question: str) -> str:
        """Build prompt for SQL generation."""
        allowed_views = ", ".join(config.allowed_views)
        
        return f"""You are a SQL query generator for data profiling statistics.

Available views:
- {allowed_views}

These views contain data profiling information with columns like:
- table_name
- column_name
- null_percentage
- distinct_count
- data_type
- row_count

Generate a SQL SELECT query to answer this question: {question}

Rules:
1. Only use SELECT statements
2. Only query from the allowed views listed above
3. Return only the SQL query, no explanations
4. Use SQLite syntax (LIMIT instead of TOP, no semicolons needed)

SQL Query:"""
    
    def _build_summary_prompt(self, question: str, results: list, explanation: str) -> str:
        """Build prompt for result summarization."""
        results_str = str(results[:10])
        
        return f"""User asked: {question}

Query returned {len(results)} results.

Data quality assessment: {explanation}

Provide a concise, natural language answer (2-3 sentences) explaining what the data shows:"""
    
    def _extract_sql_from_response(self, response: str) -> str:
        """Extract SQL query from LLM response."""
        sql_pattern = r'```(?:sql)?\s*(.*?)```'
        match = re.search(sql_pattern, response, re.DOTALL | re.IGNORECASE)
        
        if match:
            return match.group(1).strip()
        
        if 'SELECT' in response.upper():
            start = response.upper().find('SELECT')
            sql = response[start:]
            lines = sql.split('\n')
            sql_lines = []
            for line in lines:
                if line.strip() and not line.strip().startswith('```'):
                    sql_lines.append(line)
                if line.strip().endswith(';'):
                    break
            return ' '.join(sql_lines).rstrip(';').strip()
        
        return response.strip()

# Initialize Ollama client
# Note: For online environments, you may need to use a different LLM service
# or set up Ollama on a remote server
llm_client = OllamaClient(config.ollama_base_url, config.ollama_model)
print("✅ Ollama client initialized")
print(f"   Base URL: {config.ollama_base_url}")
print(f"   Model: {config.ollama_model}")


In [ ]:
# Query Router Class
class QueryRouter:
    """Routes queries to appropriate handlers (SQL vs General NLP)."""
    
    SQL_KEYWORDS = [
        'table', 'column', 'row', 'null', 'data type', 'distinct',
        'profiling', 'statistics', 'quality', 'percentage', 'count',
        'show me', 'which', 'what are', 'find', 'list', 'get',
        'query', 'select', 'database', 'schema'
    ]
    
    def route(self, question: str) -> Tuple[str, bool]:
        """Route a question to determine if it needs SQL generation."""
        question_lower = question.lower().strip()
        
        is_data_request = any([
            'which columns' in question_lower and ('null' in question_lower or 'high' in question_lower or 'percentage' in question_lower),
            'which tables' in question_lower and ('most' in question_lower or 'largest' in question_lower or 'rows' in question_lower),
            'show me' in question_lower and ('table' in question_lower or 'column' in question_lower),
            'what are the' in question_lower and ('columns' in question_lower or 'tables' in question_lower or 'data types' in question_lower) and ('in' in question_lower or 'of' in question_lower),
            'list' in question_lower and ('columns' in question_lower or 'tables' in question_lower),
            'find' in question_lower and ('columns' in question_lower or 'tables' in question_lower),
        ])
        
        explicit_sql_patterns = [
            r'\b(show|list|find|get|query|select).*\b(table|column|row|data).*\b(null|percentage|count|distinct|type)',
            r'\bwhich\s+(columns?|tables?|rows?).*\b(have|with|contain|show)',
            r'\bwhat\s+(are|is)\s+the\s+(columns?|tables?|rows?|data\s+types?)\s+(in|of|with)',
        ]
        has_explicit_sql = any(re.search(pattern, question_lower) for pattern in explicit_sql_patterns)
        
        general_indicators = [
            question_lower.startswith(('what is', 'how does', 'why', 'explain', 'tell me about', 'can you help', 'help me understand')),
            question_lower.startswith(('i want to', 'i need to', 'i\'m looking to', 'i\'d like to')),
            len(question.split()) < 5 and '?' in question,
            question_lower in ['hello', 'hi', 'hey', 'help', 'what can you do'],
        ]
        
        has_general = any(general_indicators)
        
        if (is_data_request or has_explicit_sql) and not has_general:
            return ('sql', True)
        else:
            return ('general', False)

query_router = QueryRouter()
print("✅ Query router initialized")


In [ ]:
# Data Quality Explainer Class
class DataQualityExplainer:
    """Generates rule-based explanations for data profiling results."""
    
    NULL_PERCENTAGE_HIGH = 50.0
    NULL_PERCENTAGE_MODERATE = 10.0
    DISTINCT_COUNT_LOW = 2
    
    def explain(self, results: List[Dict[str, Any]], question: str) -> str:
        """Generate rule-based explanation for query results."""
        if not results:
            return "No data found matching your query."
        
        explanations = []
        
        for row in results:
            row_explanation = self._explain_row(row)
            if row_explanation:
                explanations.append(row_explanation)
        
        if explanations:
            return " | ".join(explanations)
        
        return f"Found {len(results)} result(s). Data appears to be within normal parameters."
    
    def _explain_row(self, row: Dict[str, Any]) -> str:
        """Generate explanation for a single row of results."""
        explanations = []
        
        null_pct = self._get_numeric_value(row, 'null_percentage')
        if null_pct is not None:
            if null_pct > self.NULL_PERCENTAGE_HIGH:
                explanations.append("High null percentage indicates data quality issue")
            elif null_pct > self.NULL_PERCENTAGE_MODERATE:
                explanations.append("Moderate null percentage requires data cleaning")
            elif null_pct > 0:
                explanations.append("Low null percentage - good data quality")
            else:
                explanations.append("No null values - excellent data quality")
        
        distinct_count = self._get_numeric_value(row, 'distinct_count')
        if distinct_count is not None and distinct_count < self.DISTINCT_COUNT_LOW:
            explanations.append("Low distinct count suggests limited data diversity")
        
        table_name = row.get('table_name', '')
        column_name = row.get('column_name', '')
        
        if table_name and column_name:
            context = f"{table_name}.{column_name}"
        elif table_name:
            context = table_name
        elif column_name:
            context = column_name
        else:
            context = None
        
        if explanations and context:
            return f"{context}: {', '.join(explanations)}"
        elif explanations:
            return ', '.join(explanations)
        
        return None
    
    def _get_numeric_value(self, row: Dict[str, Any], key: str) -> Optional[float]:
        """Safely extract numeric value from row."""
        value = row.get(key)
        if value is None:
            return None
        
        try:
            return float(value)
        except (ValueError, TypeError):
            return None

explainer = DataQualityExplainer()
print("✅ Data quality explainer initialized")


In [ ]:
# General NLP Handler Class
class GeneralNLPHandler:
    """Handles general NLP queries that don't require SQL generation."""
    
    def __init__(self, base_url: str, model: str):
        self.base_url = base_url
        self.model = model
    
    def process_query(self, question: str, context: Optional[dict] = None) -> str:
        """Process a general NLP query."""
        prompt = self._build_general_prompt(question, context)
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.8,
                        "top_p": 0.95,
                    }
                },
                timeout=120
            )
            response.raise_for_status()
            
            result = response.json()
            return result.get("response", "").strip()
            
        except Exception as e:
            logger.error(f"Ollama API error: {e}")
            return "I apologize, but I'm having trouble processing your request right now. Please try again."
    
    def _build_general_prompt(self, question: str, context: Optional[dict] = None) -> str:
        """Build prompt for general NLP queries."""
        base_prompt = f"""You are a friendly and knowledgeable data profiling assistant. You help users understand data quality, analytics, and answer questions in a natural, conversational way.

User question: {question}
"""
        
        if context:
            context_info = f"\nContext:\n"
            if context.get('available_tables'):
                context_info += f"Available tables: {', '.join(context['available_tables'])}\n"
            if context.get('database_info'):
                context_info += f"Database: {context['database_info']}\n"
            base_prompt += context_info
        
        base_prompt += """
Instructions:
- Be conversational, friendly, and helpful
- Answer naturally as if you're having a conversation
- If asked about data profiling concepts, explain them clearly with examples
- If the question is about specific data in the database, you can offer to help query it
- Feel free to ask clarifying questions if needed
- Keep responses concise but informative
- You can discuss data quality, analytics, best practices, and general data topics

Provide a natural, helpful response:"""
        
        return base_prompt

general_nlp = GeneralNLPHandler(config.ollama_base_url, config.ollama_model)
print("✅ General NLP handler initialized")


## Step 4: Main Chat Function


In [ ]:
def chat(question: str) -> Dict[str, Any]:
    """
    Main chat function that handles both SQL queries and general NLP questions.
    
    Args:
        question: User's question
        
    Returns:
        Dictionary with answer, sql_query, results_count, and explanation
    """
    question = question.strip()
    
    if not question:
        return {
            "answer": "Please provide a question.",
            "sql_query": None,
            "results_count": 0,
            "explanation": None
        }
    
    logger.info(f"Received question: {question}")
    
    # Route the query
    query_type, needs_sql = query_router.route(question)
    logger.info(f"Query routed as: {query_type} (needs_sql: {needs_sql})")
    
    try:
        if needs_sql:
            return _handle_sql_query(question)
        else:
            return _handle_general_query(question)
    except Exception as e:
        logger.error(f"Unexpected error: {e}", exc_info=True)
        return {
            "answer": f"Error: {str(e)}",
            "sql_query": None,
            "results_count": 0,
            "explanation": None
        }


def _handle_sql_query(question: str) -> Dict[str, Any]:
    """Handle SQL query generation and execution."""
    # Step 1: Generate SQL from natural language
    sql_query = llm_client.generate_sql(question)
    
    if not sql_query:
        return {
            "answer": "Failed to generate SQL query. Please check Ollama connection.",
            "sql_query": None,
            "results_count": 0,
            "explanation": None
        }
    
    # Step 2: Validate SQL query
    is_valid, error_message = sql_validator.validate(sql_query)
    
    if not is_valid:
        logger.warning(f"SQL validation failed: {error_message}")
        return {
            "answer": f"Generated SQL query is not safe: {error_message}",
            "sql_query": sql_query,
            "results_count": 0,
            "explanation": None
        }
    
    # Sanitize query
    sql_query = sql_validator.sanitize(sql_query)
    
    # Step 3: Execute query
    try:
        results = db.execute_query(sql_query)
    except Exception as e:
        logger.error(f"Query execution failed: {e}")
        return {
            "answer": f"Database query failed: {str(e)}",
            "sql_query": sql_query,
            "results_count": 0,
            "explanation": None
        }
    
    # Step 4: Generate rule-based explanation
    explanation = explainer.explain(results, question)
    
    # Step 5: Generate natural language summary
    answer = llm_client.summarize_results(question, results, explanation)
    
    # Fallback to explanation if LLM summarization fails
    if not answer or len(answer.strip()) < 10:
        answer = explanation
    
    logger.info(f"Successfully processed SQL question. Returned {len(results)} results")
    
    return {
        "answer": answer,
        "sql_query": sql_query,
        "results_count": len(results),
        "explanation": explanation,
        "results": results[:10]  # Include first 10 results for display
    }


def _handle_general_query(question: str) -> Dict[str, Any]:
    """Handle general NLP queries with conversational responses."""
    # Get context about available data (optional, non-blocking)
    context = {}
    try:
        tables_result = db.execute_query(
            "SELECT DISTINCT table_name FROM profiling_table_stats LIMIT 10"
        )
        context['available_tables'] = [row['table_name'] for row in tables_result]
        context['database_info'] = "Sample profiling database with data quality metrics"
    except:
        pass  # Continue without context if database unavailable
    
    # Process with general NLP
    answer = general_nlp.process_query(question, context)
    
    logger.info("Successfully processed general NLP question")
    
    return {
        "answer": answer,
        "sql_query": None,
        "results_count": 0,
        "explanation": None
    }

print("✅ Chat functions ready")


## Step 5: Interactive Chat Interface

You can now use the chatbot! Try asking questions like:
- "Which columns have high null percentages?"
- "Show me the tables with the most rows"
- "What are the data types in the customers table?"
- "Explain data profiling"


In [ ]:
# Interactive chat function with pretty printing
def ask_question(question: str):
    """Ask a question and display the response in a formatted way."""
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}\n")
    
    response = chat(question)
    
    print(f"Answer: {response['answer']}\n")
    
    if response['sql_query']:
        print(f"SQL Query: {response['sql_query']}\n")
    
    if response['results_count'] > 0:
        print(f"Results: {response['results_count']} row(s) found")
        if 'results' in response and response['results']:
            print("\nFirst few results:")
            import pandas as pd
            df = pd.DataFrame(response['results'])
            display(df.head(10))
    
    if response['explanation']:
        print(f"\nExplanation: {response['explanation']}")
    
    print(f"\n{'='*60}\n")
    
    return response

# Test the chatbot
print("Chatbot is ready! Use ask_question('your question here') to interact.")


In [ ]:
# Example usage
ask_question("Which columns have high null percentages?")


In [ ]:
# Another example
ask_question("Show me the tables with the most rows")


## Optional: Create a Simple Widget Interface

For a more interactive experience, you can use IPython widgets:


In [ ]:
# Install ipywidgets if not already installed
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    print("ipywidgets not available. Install with: !pip install ipywidgets")
    WIDGETS_AVAILABLE = False

if WIDGETS_AVAILABLE:
    # Create interactive chat widget
    chat_output = widgets.Output()
    
    question_input = widgets.Text(
        value='',
        placeholder='Type your question here...',
        description='Question:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )
    
    submit_button = widgets.Button(
        description='Ask',
        button_style='primary',
        layout=widgets.Layout(width='15%', margin='0 0 0 2%')
    )
    
    def on_submit(b):
        with chat_output:
            clear_output(wait=True)
            question = question_input.value
            if question:
                ask_question(question)
            question_input.value = ''
    
    submit_button.on_click(on_submit)
    
    # Display the widget
    display(widgets.VBox([
        widgets.HBox([question_input, submit_button]),
        chat_output
    ]))
    
    print("✅ Interactive chat widget is ready!")
else:
    print("Use ask_question('your question') function to interact with the chatbot.")


## Notes for Online Environments

### Using with Google Colab or JupyterHub

1. **Ollama Setup**: For online environments, you have a few options:
   - Use a remote Ollama server (update `config.ollama_base_url`)
   - Use alternative LLM services (OpenAI, Anthropic, etc.) - modify `OllamaClient` class
   - Use Hugging Face Transformers for local inference

2. **Database**: This notebook uses SQLite which works perfectly in online environments.

3. **Dependencies**: All required packages are installed in the first cell.

### Alternative LLM Services

If you want to use a different LLM service, you can modify the `OllamaClient` class to use:
- OpenAI API
- Anthropic Claude API
- Hugging Face Inference API
- Google Gemini API

Just update the API calls in the `generate_sql` and `summarize_results` methods.
